### GPT 1

In [ ]:
# dependencies 
import math 
import torch
import torch.nn as nn 
import torch.nn.functional as F
from pydantic import BaseModel, Field, model_validator

In [ ]:
# Configuration
class GPT1Config(BaseModel):
    vocab_size: int = Field(default = 40478, gt = 0, description = "Vocabulary size")

    cotext_length: int = Field(default = 572, gt = 0, description = "Max cotext length/Block size")
    hidden_size: int = Field(default = 768, gt = 0, description = " model/hidden dimension aka d_model") # embed_dim == hidden_size == n_embed == d_model
    n_layer: int = Field(default = 12, gt = 0, description = "number of stacked decoder blocks")
    num_head: int = Field(default = 12, gt = 0, description = "number of attention heads")
    intermediate_step: int = Field(default = 3072, gt = 0, description = "FFN inner dimension")
    dropout: float = Field(default = 0.1, ge = 0.0, description = "Residual dropout")
    attention_dropout: float = Field(default = 0.1, ge = 0.0, description = "attention dropout")
    embedding_dropout: float = Field(default = 0.1, ge = 0.0, description = "embedding dropout")

    model_config = {"freeze": True} 

    @model_validator(mode = "after") # operates over the entire model input, whereas the field_validator only operates on an isolated class where it define
    def check_head_divide_embed(self):
        if self.n_embed % self.num_head != 0:
            raise ValueError(
                f"n_embed ({self.n_embed} must be divisable by num_heads ({self.num_head});"
                f"got remainder {self.n_embed % self.num_head}"
                )
        return self 

In [ ]:
# Masked MultiHeadSelfAttention(nn.Module)
class MaskedMultiHeadSelfAttention(nn.Module):
    """
            Shapes (B = batch, T = sequence length, C = hidden_size, H = n_head, D = C/H):
            input  x        : (B, T, C)
            qkv projection   -> (B, T, 3C)  -> split into q,k,v : (B, T, C) each
            reshape to heads : (B, H, T, D)
            attention scores : (B, H, T, T)
            context          : (B, H, T, D) -> merge heads -> (B, T, C)
            output projection: (B, T, C)
        """
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.num_head = config.num_head
        self.head_dim = config.hidden_size/config.num_head

        # only one matmul can produce Q, K, V together (faster than separate 3)
        self.qkv_proj = nn.Linear(config.hidden_size, 3 * config.hidden_size)
        self.out_ptoj = nn.Linear(config.hidden_size, config.hidden_size)

        self.attention_dropout = nn.Dropout(config.attention_dropout)
        self.residual_dropout = nn.Dropout(config.dropout)

        # perform the lower trianglular part
        causal_mask = torch.tril(torch.ones(config.context_len, config.context_length))
        
        self.register_buffer("causal_mask", causal_mask.view(1, 1, config.context_length, config.context_length))

    def forward(self, x):
        B, T, C = x.shape # [batch_size, seq_len, hidden_size]

        qkv = self.qkv_proj(x) # [B, T, 3C]
        q, k, v = qkv.split(C, dim = 2) # [B, T, C]

        # [B, H, T, D]
        q = q.view(B, T, self.num_head, self.head_dim).tranpose(1, 2)
        k = k.view(B, T, self.num_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_head, self.head_dim).tranpose(1,2)

        # scaled dot-product attention
        attention = q @ k.transpose(-2, -1) # [B, H, T, D] @ [B, H, D, T] --> [B, H, T, T]
        attention = attention / math.sqrt(self.head_dim) # scaling
        attention = attention.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        attention = F.softmax(attention, dim = -1)
        attention = self.attention_dropout(attention)

        score = attention @ v
        score = score.transpose(1,2).contiguous().view(B, T, C) # merge the head (num_head * )

        return self.residual_dropout(self.out_proj(score))

In [ ]:
# Position-wise FFN 
class PositionWiseFNN(nn.Module):
    # (B,T,C) -> Linear: (B,T,d_ff) -> Activation: GELU -> Linear: (B,T,C)
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.fc1 = nn.Linear(config.hidden_size, config.intermediate_step)
        self.fc2 = nn.Linear(config.intermediate_step, config.hidden_size)
        self.dropout = nn.DropOut(config.dropout)
        # nn.Linear only cares about the last dim of the tensor

    def forward(self):
        x = self.fc1(x)
        x = F.GELU(x)
        x = self.fc2(x)
        return self.dropout(x)

In [ ]:
# 1 Decoder Block 
class TransformerBlock(nn.Module):
    def __init__(self, config: GPT1Config):
        super().__init__()
        self.attention = MaskedMultiHeadSelfAttention(config)
        self.ln1 = nn.LayerNorm(config.hidden_size)
        self.ffn = PositionWiseFNN(config)
        self.ln2 = nn.LayerNorm(config.hidden_size)

    # Post-norm technique
    def forward(self, x):
        x = self.ln1(x + self.attention(x))
        x = self.ln2(x + self.ffn(x))
        return x